# Setup

## Import Lib

In [20]:
import os
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import sklearn

from urllib.parse import urlparse
from sklearn.ensemble import RandomForestClassifier

## Path

In [21]:
GITHUB_USERNAME = "Rashifalmas"
REPO_NAME = "Pedas2026"
BRANCH = "main"

BASE_URL = (
    f"https://raw.githubusercontent.com/"
    f"{GITHUB_USERNAME}/{REPO_NAME}/{BRANCH}/data"
)

TRAIN_URL = f"{BASE_URL}/training.csv"
PREDICT_URL = f"{BASE_URL}/predict.csv"

print("=" * 80)
print("GITHUB DATA SOURCE")
print("=" * 80)
print("Training URL :", TRAIN_URL)
print("Prediction URL:", PREDICT_URL)

GITHUB DATA SOURCE
Training URL : https://raw.githubusercontent.com/Rashifalmas/Pedas2026/main/data/training.csv
Prediction URL: https://raw.githubusercontent.com/Rashifalmas/Pedas2026/main/data/predict.csv


## Load

In [22]:
train = pd.read_csv(TRAIN_URL)
predict = pd.read_csv(PREDICT_URL)

# Rapikan nama kolom
train.columns = train.columns.str.strip()
predict.columns = predict.columns.str.strip()

print("\n" + "=" * 80)
print("DATA LOADED SUCCESSFULLY")
print("=" * 80)

print("Train shape  :", train.shape)
print("Predict shape:", predict.shape)

print("\nTraining columns:")
print(train.columns.tolist())

print("\nPrediction columns:")
print(predict.columns.tolist())

# ============================================================
# VALIDATION
# ============================================================

assert train.shape[0] == 8400, (
    f"Jumlah baris training tidak sesuai: {train.shape[0]}"
)

assert predict.shape[0] == 1500, (
    f"Jumlah baris prediction tidak sesuai: {predict.shape[0]}"
)

assert "category" in train.columns, (
    "Kolom 'category' tidak ditemukan di dataset training."
)

assert "id" in predict.columns, (
    "Kolom 'id' tidak ditemukan di dataset prediction."
)

print("\nDataset validation: PASSED")

print("\nscikit-learn:", sklearn.__version__)

if tuple(map(int, sklearn.__version__.split(".")[:2])) < (1, 4):
    raise RuntimeError(
        "scikit-learn terlalu lama. "
        "Canonical A4 membutuhkan native NaN support."
    )


DATA LOADED SUCCESSFULLY
Train shape  : (8400, 10)
Predict shape: (1500, 10)

Training columns:
['url', 'brand', 'discovered', 'confidence_level', 'ip', 'domain', 'sld', 'category', 'registrar', 'registration_date']

Prediction columns:
['id', 'url', 'brand', 'discovered', 'confidence_level', 'ip', 'domain', 'sld', 'registrar', 'registration_date']

Dataset validation: PASSED

scikit-learn: 1.6.1


# Data Prep

## LABEL NORMALIZATION CANONICAL

In [23]:
LABEL_MAP = {
    "online gambling": "online gambling",
    "Online Gambling": "online gambling",
    "online gamblingg": "online gambling",

    "phishing": "phishing",
    "phishingg": "phishing",

    "other": "other",
    "Other": "other",
    "otherr": "other",

    "spam": "spam",
    "spamm": "spam",

    "malware": "malware",
    "malwaree": "malware",

    "Brand": "brand",
    "brand": "brand",

    "FakeShop": "fakeshop",
    "fakeshop": "fakeshop",

    "violence": "violence",

    "PIIExposure": "piiexposure",
    "piiexposure": "piiexposure",
}

raw_labels = set(train["category"].unique())

unmapped = raw_labels - set(LABEL_MAP.keys())

if unmapped:
    raise ValueError(
        f"Label belum ada dalam LABEL_MAP: {unmapped}"
    )

train["category"] = train["category"].map(LABEL_MAP)

assert train["category"].nunique() == 9

print("\nNormalized labels:")
print(train["category"].value_counts())


Normalized labels:
category
online gambling    5447
phishing           2253
other               284
spam                185
malware             179
brand                45
fakeshop              5
violence              1
piiexposure           1
Name: count, dtype: int64


# Feature Extraction

## TARGET

In [24]:
TARGET_COL = "category"

y = train[TARGET_COL].copy()

ALL_LABELS = sorted(
    train[TARGET_COL].unique()
)

print("\nLabels:")
print(ALL_LABELS)


Labels:
['brand', 'fakeshop', 'malware', 'online gambling', 'other', 'phishing', 'piiexposure', 'spam', 'violence']


## BASE FEATURES — CANONICAL

In [25]:
SLD_COL = "sld"
CONF_COL = "confidence_level"
URL_COL = "url"
IP_COL = "ip"
DOMAIN_COL = "domain"
DISCOVERED_COL = "discovered"
REG_DATE_COL = "registration_date"
BRAND_COL = "brand"

required_cols = [
    SLD_COL,
    CONF_COL,
    URL_COL,
    IP_COL,
    DOMAIN_COL,
    DISCOVERED_COL,
    REG_DATE_COL,
    BRAND_COL,
]

for col in required_cols:
    assert col in train.columns, (
        f"Kolom {col} tidak ditemukan."
    )

In [26]:
train_url = train[URL_COL].astype(str)

train["url_len"] = train_url.str.len()

# ------------------------------------------------------------
# datetime
# ------------------------------------------------------------

train_discovered = pd.to_datetime(
    train[DISCOVERED_COL],
    errors="coerce"
)

train_registration = pd.to_datetime(
    train[REG_DATE_COL],
    errors="coerce"
)

invalid_discovered = (
    train[DISCOVERED_COL].notna()
    & train_discovered.isna()
).sum()

invalid_registration = (
    train[REG_DATE_COL].notna()
    & train_registration.isna()
).sum()

assert invalid_discovered == 0
assert invalid_registration == 0


# IMPORTANT:
# canonical = .dt.days
train["domain_age"] = (
    train_discovered - train_registration
).dt.days


# ------------------------------------------------------------
# SLD OHE
# ------------------------------------------------------------

X_base = pd.get_dummies(
    train[
        [
            CONF_COL,
            SLD_COL,
            "url_len",
            "domain_age",
        ]
    ],
    columns=[SLD_COL],
    prefix=SLD_COL,
)

print("\nBase features:", X_base.shape)

assert X_base.shape[1] == 16


Base features: (8400, 16)


## CANONICAL has_brand, brand_na

In [27]:
train["has_brand"] = (
    train[BRAND_COL].notna()
    & (train[BRAND_COL] != "-")
).astype(int)

train["brand_na"] = (
    train[BRAND_COL].isna()
).astype(int)

## URL FEATURES

In [28]:
def extract_url_features(series):

    s = series.astype(str)

    result = pd.DataFrame(index=series.index)

    parsed = s.apply(urlparse)

    # host / netloc
    host = parsed.apply(
        lambda x: x.netloc.lower()
        if x.netloc
        else ""
    )

    path = parsed.apply(
        lambda x: x.path
        if x.path
        else ""
    )

    query = parsed.apply(
        lambda x: x.query
        if x.query
        else ""
    )

    scheme = parsed.apply(
        lambda x: x.scheme.lower()
        if x.scheme
        else ""
    )

    # --------------------------------------------------------
    # URL lexical features
    # --------------------------------------------------------

    result["scheme_none"] = (
        scheme == ""
    ).astype(int)

    result["scheme_https"] = (
        scheme == "https"
    ).astype(int)

    result["has_path"] = (
        path.str.len() > 0
    ).astype(int)

    result["n_dots_host"] = (
        host.str.count(r"\.")
    )

    result["path_len"] = (
        path.str.len()
    )

    result["has_www"] = (
        host.str.startswith("www.")
    ).astype(int)

    result["has_query"] = (
        query.str.len() > 0
    ).astype(int)

    result["n_digits"] = (
        s.str.count(r"\d")
    )

    result["has_at"] = (
        s.str.contains("@", regex=False)
    ).astype(int)

    result["n_hyphen"] = (
        s.str.count("-")
    )

    result["host_len"] = (
        host.str.len()
    )

    result["n_slash"] = (
        s.str.count("/")
    )

    return result


X_url = extract_url_features(
    train[URL_COL]
)

print("\nURL features:", X_url.shape)

assert X_url.shape[1] == 12


URL features: (8400, 12)


## TEMPORAL FEATURES

In [29]:
X_time = pd.DataFrame(
    index=train.index
)

X_time["disc_hour"] = (
    train_discovered.dt.hour
)

X_time["disc_minute"] = (
    train_discovered.dt.minute
)

X_time["is_midnight"] = (
    (train_discovered.dt.hour == 0)
    & (train_discovered.dt.minute == 0)
).astype(int)

# canonical timestamp
X_time["disc_ts"] = (
    train_discovered.astype("int64")
    / 10**9
)

X_time["disc_dow"] = (
    train_discovered.dt.dayofweek
)

print("\nTemporal features:", X_time.shape)

assert X_time.shape[1] == 5


Temporal features: (8400, 5)


# FULL-TRAIN

In [30]:
def full_frequency_encode(
    train_series,
    predict_series
):

    freq_map = (
        train_series
        .dropna()
        .value_counts()
        .to_dict()
    )

    train_encoded = train_series.map(
        freq_map
    )

    predict_encoded = predict_series.map(
        freq_map
    )

    # nilai non-null tetapi tidak dikenal
    train_unseen = (
        train_series.notna()
        & train_encoded.isna()
    )

    predict_unseen = (
        predict_series.notna()
        & predict_encoded.isna()
    )

    train_encoded.loc[train_unseen] = 0
    predict_encoded.loc[predict_unseen] = 0

    return train_encoded, predict_encoded


train_ip_freq, _ = full_frequency_encode(
    train[IP_COL],
    predict[IP_COL]
)

train_domain_freq, _ = full_frequency_encode(
    train[DOMAIN_COL],
    predict[DOMAIN_COL]
)


X_freq = pd.DataFrame({
    "ip_frequency": train_ip_freq,
    "domain_frequency": train_domain_freq,
})

print("\nFrequency features:", X_freq.shape)


Frequency features: (8400, 2)


# COMBINE CANONICAL A4 TRAIN

In [31]:
X_train = pd.concat(
    [
        X_base,
        pd.DataFrame({
            "has_brand": train["has_brand"],
            "brand_na": train["brand_na"],
        }),
        X_url,
        X_time,
        X_freq,
    ],
    axis=1
)

print("\n" + "=" * 80)
print("CANONICAL A4 TRAIN")
print("=" * 80)

print("Shape:", X_train.shape)

assert X_train.shape[1] == 37
assert X_train.columns.duplicated().sum() == 0


CANONICAL A4 TRAIN
Shape: (8400, 37)


# BUILD PREDICT FEATURES

In [32]:
predict_url = predict[URL_COL].astype(str)

predict_discovered = pd.to_datetime(
    predict[DISCOVERED_COL],
    errors="coerce"
)

predict_registration = pd.to_datetime(
    predict[REG_DATE_COL],
    errors="coerce"
)

predict_url_len = (
    predict_url.str.len()
)

predict_domain_age = (
    predict_discovered
    - predict_registration
).dt.days

In [33]:
# ------------------------------------------------------------
# SLD OHE
# ------------------------------------------------------------

X_predict_base = pd.get_dummies(
    predict[
        [
            CONF_COL,
            SLD_COL,
        ]
    ],
    columns=[SLD_COL],
    prefix=SLD_COL,
)

X_predict_base["url_len"] = (
    predict_url_len
)

X_predict_base["domain_age"] = (
    predict_domain_age
)

# EXACT same columns as train
X_predict_base = X_predict_base.reindex(
    columns=X_base.columns,
    fill_value=0
)


# ------------------------------------------------------------
# Brand
# ------------------------------------------------------------

predict_has_brand = (
    predict[BRAND_COL].notna()
    & (predict[BRAND_COL] != "-")
).astype(int)

predict_brand_na = (
    predict[BRAND_COL].isna()
).astype(int)

X_predict_brand = pd.DataFrame({
    "has_brand": predict_has_brand,
    "brand_na": predict_brand_na,
}, index=predict.index)


# ------------------------------------------------------------
# URL
# ------------------------------------------------------------

X_predict_url = extract_url_features(
    predict[URL_COL]
)


# ------------------------------------------------------------
# Temporal
# ------------------------------------------------------------

X_predict_time = pd.DataFrame(
    index=predict.index
)

X_predict_time["disc_hour"] = (
    predict_discovered.dt.hour
)

X_predict_time["disc_minute"] = (
    predict_discovered.dt.minute
)

X_predict_time["is_midnight"] = (
    (predict_discovered.dt.hour == 0)
    & (predict_discovered.dt.minute == 0)
).astype(int)

X_predict_time["disc_ts"] = (
    predict_discovered.astype("int64")
    / 10**9
)

X_predict_time["disc_dow"] = (
    predict_discovered.dt.dayofweek
)


# ------------------------------------------------------------
# Frequency
# ------------------------------------------------------------

_, predict_ip_freq = full_frequency_encode(
    train[IP_COL],
    predict[IP_COL]
)

_, predict_domain_freq = full_frequency_encode(
    train[DOMAIN_COL],
    predict[DOMAIN_COL]
)

X_predict_freq = pd.DataFrame({
    "ip_frequency": predict_ip_freq,
    "domain_frequency": predict_domain_freq,
}, index=predict.index)

# COMBINE PREDICT

In [34]:
X_predict = pd.concat(
    [
        X_predict_base,
        X_predict_brand,
        X_predict_url,
        X_predict_time,
        X_predict_freq,
    ],
    axis=1
)

# force identical column order
X_predict = X_predict.reindex(
    columns=X_train.columns
)

print("\n" + "=" * 80)
print("CANONICAL A4 PREDICT")
print("=" * 80)

print("Shape:", X_predict.shape)

assert X_predict.shape == (
    1500,
    37
)

assert list(X_train.columns) == list(
    X_predict.columns
)


CANONICAL A4 PREDICT
Shape: (1500, 37)


In [35]:
# ============================================================
# 13. SANITY CHECK FEATURES
# ============================================================

print("\nMissing values:")
print(
    X_train.isna().sum()
    [lambda x: x > 0]
)


Missing values:
domain_age        21
ip_frequency    1257
dtype: int64


# RANDOM FOREST — CANONICAL

In [36]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight=None,
    random_state=42,
    n_jobs=-1,
)

print("\n" + "=" * 80)
print("TRAINING CANONICAL A4 RF100")
print("=" * 80)

model.fit(
    X_train,
    y
)

pred = model.predict(
    X_predict
)

pred = pd.Series(
    pred,
    name="category"
)


TRAINING CANONICAL A4 RF100


# PREDICTION DISTRIBUTION

In [37]:
print("\n" + "=" * 80)
print("CANONICAL A4 PREDICTION DISTRIBUTION")
print("=" * 80)

distribution = (
    pred
    .value_counts()
    .reindex(ALL_LABELS, fill_value=0)
)

print(distribution)


CANONICAL A4 PREDICTION DISTRIBUTION
category
brand                9
fakeshop             0
malware             28
online gambling    984
other               48
phishing           399
piiexposure          0
spam                32
violence             0
Name: count, dtype: int64
